# Basic RAG System Notebook

This project demonstrates a complete Retrieval-Augmented Generation system using Wikipedia retrieval and transformer-based answer generation with evaluation metrics and deployment readiness.

## Import required libraries

These imports support the complete RAG workflow, including dataset loading, Wikipedia-based retrieval, transformer-based answer generation, and evaluation preprocessing.

In [11]:
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
import wikipedia

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


## Configuration & Hyperparameters
This section defines key configuration parameters such as dataset size and generation model selection.


In [12]:
MAX_SAMPLES = 100
GENERATION_MODEL = "google/flan-t5-base"


## Load FLAN-T5 Generation Model

This block loads the tokenizer and FLAN-T5 generation model used for context-grounded answer generation

In [13]:
tokenizer = AutoTokenizer.from_pretrained(GENERATION_MODEL)
model_gen = AutoModelForSeq2SeqLM.from_pretrained(GENERATION_MODEL)

print("Generation model loaded successfully.")


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 4242.14it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Generation model loaded successfully.


## Load Natural Questions Dataset

This section loads the Natural Questions dataset from JSONL format and converts it into a structured DataFrame for evaluation.

In [ ]:
DATA_PATH = "NQ-open.dev.jsonl"

def load_jsonl(path, max_samples=None):

    data = []

    with open(path, 'r', encoding='utf-8') as f:

        for idx, line in enumerate(f):

            if max_samples and idx >= max_samples:
                break

            data.append(json.loads(line))

    return pd.DataFrame(data)

df = load_jsonl(DATA_PATH, MAX_SAMPLES)

# This preprocessing step standardizes answer formats to improve evaluation consistency.

def normalize_answer(ans):

    if isinstance(ans, list):
        return ans[0]

    return ans

df['answer'] = df['answer'].apply(normalize_answer)

df.head()


,question,answer
0,when was the last time anyone was on the moon,14 December 1972 UTC
1,who wrote he ain't heavy he's my brother lyrics,Bobby Scott
2,how many seasons of the bastard executioner ar...,one
3,when did the eagles win last super bowl,2017
4,who won last year's ncaa women's basketball,South Carolina


## Wikipedia Retrieval Function
This function implements the retrieval stage of the RAG pipeline by dynamically fetching relevant Wikipedia documents.


In [ ]:
def retrieve_wikipedia_context(query):

    documents = [] # Store retrieved documents
  
  # Perform Wikipedia search using the user query.This retrieves semantically related page titles.
    try: 

        search_results = wikipedia.search(query, results=5)

        for title in search_results:  # Loop through retrieved Wikipedia page titles

            try: # Retrieve page content for each title

                page = wikipedia.page(
                    title,
                    auto_suggest=False
                )

                documents.append(page.summary)

            except:
                continue
 # Handle Wikipedia search failures
    except:
        pass

  # Fallback when no documents are retrieved
    if len(documents) == 0:

        documents.append(
            f"No information found for {query}"
        )

    return "\n".join(documents)


## RAG Generation Function

This block combines retrieved context with the user query and uses FLAN-T5 to generate grounded answers.

In [ ]:
def generate_answer(question):

    context = retrieve_wikipedia_context(question)     # Retrieve relevant Wikipedia documents using the user question

# Construct prompt for the LLM : The retrieved context is added before the question so that FLAN-T5 generates grounded answers
    prompt = f"""
Answer the question briefly using the context below.

Context:
{context}

Question:
{question}

Answer:
"""
 # Convert text prompt into tokens understandable by the model
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512  
    )
# Generate output tokens
    outputs = model_gen.generate(
        **inputs,
        max_new_tokens=32,  # maximum generated answer length
        do_sample=False # the model gives the SAME output every time
    )

# Decode generated token IDs back into human-readable text

    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
 # Return:
    # 1. Generated answer
    # 2. Retrieved context
    return answer, context


## Test the RAG Pipeline

This section tests the end-to-end RAG pipeline on a sample question.

In [17]:
question = "Who invented Python programming language?"

prediction, context = generate_answer(question)

print("QUESTION:\n")
print(question)

print("\nANSWER:\n")
print(prediction)

print("\nRETRIEVED CONTEXT:\n")
print(context[:1000])


QUESTION:

Who invented Python programming language?

ANSWER:

No information found for

RETRIEVED CONTEXT:

No information found for Who invented Python programming language?


## Text Cleaning and Metrices for Evaluation

In [ ]:
# This function preprocesses text for fair metric computation by removing punctuation and normalizing formatting.
def clean_text(text): 

    text = str(text).lower() # Convert input into string and lowercase all characters
 
    # Remove all characters except:
    # - lowercase alphabets
    # - numbers
    # - spaces
    text = re.sub(
        r"[^a-z0-9 ]",
        " ",
        text
    )
    # Replace multiple spaces with a single space
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()
    # Return cleaned text

    return text

# Exact Match measures whether the generated answer exactly matches the ground truth answer.
def exact_match(prediction, ground_truth):

    pred = clean_text(prediction)

    gt = clean_text(ground_truth)

    return int(pred == gt)

# F1 Score evaluates token-level overlap between generated and ground-truth answers.
def f1_score(prediction, ground_truth):

    pred_tokens = clean_text(prediction).split()

    gt_tokens = clean_text(ground_truth).split()

    common = set(pred_tokens) & set(gt_tokens)

    if len(common) == 0:
        return 0

    precision = len(common) / len(pred_tokens)

    recall = len(common) / len(gt_tokens)

    return (2 * precision * recall) / (precision + recall)

# Retrieval Hit Rate checks whether the correct answer exists inside the retrieved context.
def retrieval_hit(context, answer):

    context = clean_text(context)

    answer = clean_text(answer)

    return int(answer in context)


## Full Evaluation Pipeline

This section evaluates the complete RAG pipeline using Exact Match, F1 Score, and Retrieval Hit Rate.

In [19]:
def evaluate(
    questions,
    answers,
    num_samples=15
):

    correct = 0

    exact_matches = []

    f1_scores = []

    retrieval_hits = []

    for i in tqdm(range(num_samples)):

        q = questions[i]

        gt = answers[i]

        # Generate answer
        pred, context = generate_answer(q)

        # Accuracy
        if clean_text(gt) in clean_text(pred):

            correct += 1

        # Exact Match
        em = exact_match(
            pred,
            gt
        )

        # F1 Score
        f1 = f1_score(
            pred,
            gt
        )

        # Retrieval Hit
        hit = retrieval_hit(
            context,
            gt
        )

        exact_matches.append(em)

        f1_scores.append(f1)

        retrieval_hits.append(hit)

    # =====================================
    # FINAL METRICS
    # =====================================

    accuracy = correct / num_samples

    avg_em = sum(exact_matches) / num_samples

    avg_f1 = sum(f1_scores) / num_samples

    avg_hit = sum(retrieval_hits) / num_samples

    print("\n========== FINAL RESULTS ==========")


    print(f"Average Exact Match: {round(avg_em, 4)}")

    print(f"Average F1 Score: {round(avg_f1, 4)}")

    print(f"Retrieval Hit Rate: {round(avg_hit, 4)}")

## Run Evaluation

In [20]:
evaluate(
    questions=df["question"].tolist(),
    answers=df["answer"].tolist(),
    num_samples= 50
)


100%|██████████| 50/50 [03:44<00:00,  4.49s/it]


========== FINAL RESULTS ==========
Average Exact Match: 0.0
Average F1 Score: 0.0238
Retrieval Hit Rate: 0.14


### The Basic RAG pipeline produced relatively low Exact Match and F1 scores due to limitations of direct Wikipedia retrieval and lightweight generation models. However, these results motivated the transition toward Advanced RAG techniques such as semantic embeddings, FAISS vector retrieval, and reranking to improve retrieval grounding and response relevance.
